# Plots

In [ ]:
import numpy as np
import scipy.io
import scipy.linalg as la
import matplotlib.pyplot as plt

## Setup

In [ ]:
def savefig(filename, fig=None):
    if fig is None:
        fig = plt.gcf()
    fig.savefig(filename, dpi=300, bbox_inches="tight", pad_inches=0.05)

In [ ]:
# matplotlib configuration.
plt.rcParams.update(plt.rcParamsDefault)
plt.rc("axes.spines", right=False, top=False)
plt.rc("figure", dpi=300)
plt.rc("font", size=16, family="serif")
plt.rc("legend", frameon=False)
plt.rc("text", usetex=True)
plt.rc("text.latex", preamble=r"\usepackage{amsmath}")

## Figure

In [ ]:
# Load the .mat file
data = scipy.io.loadmat("plot_data.mat")

# Extract the data
x = data["x"].flatten()  # Flatten to convert from 2D array to 1D array
z_hifi = data["z_hifi"].flatten()
z_lofi = data["z_lofi"].flatten()
z_after = data["z_update_mean"].flatten()
u_hifi = data["u_hifi"].flatten()
u_lofi = data["u_lofi"].flatten()
u_before = data["u_before"].flatten()
u_after = data["u_after"].flatten()
T = data["T"].flatten()

# Create the figure
fig, axes = plt.subplots(1, 2, figsize=(12, 2.5), sharex=True)

# First subplot: Controls
axes[0].plot(
    x,
    z_hifi,
    linestyle="--",
    linewidth=1.5,
    color="C0",
    label=r"$z^*$",
)
axes[0].plot(
    x,
    z_lofi,
    linestyle="-.",
    linewidth=1.5,
    color="C1",
    label=r"$\tilde{z}$",
)
axes[0].plot(
    x,
    z_after,
    linestyle="-",
    linewidth=1.5,
    color="C2",
    label=r"$z'$",
)
# axes[0].set_title("Controls", y=0.9)
axes[0].set_xlabel(r"$x$")
axes[0].set_ylabel(r"Control")
# axes[0].legend(loc="best")

# Add annotations for the lines
axes[0].text(
    x[15],
    z_hifi[0],
    r"true solution $\mathbf{z}^*$",
    color="C0",
    fontsize="small",
    ha="left",
    va="center",
)
axes[0].text(
    x[5],
    z_lofi[0] * 0.95,
    r"surrogate solution $\tilde{\mathbf{z}}$",
    color="C1",
    fontsize="small",
    ha="left",
    va="top",
)
axes[0].text(
    x[100],
    z_after[100],
    r"updated solution $\mathbf{z}'$",
    color="C2",
    fontsize="small",
    ha="left",
    va="bottom",
)

# Second subplot: States
axes[1].plot(
    x,
    T,
    "k-",
    linewidth=1.5,
    label=r"$\phi$",
)
axes[1].text(
    x[35],
    T[40],
    r"target $\phi$",
    color="k",
    fontsize="small",
    ha="right",
    va="bottom",
)
axes[1].plot(
    x,
    u_hifi,
    "C0--",
    linewidth=1.5,
    label=r"$\mathbf{s}(\mathbf{z}^*)$",
)
axes[1].plot(
    x,
    u_before,
    linestyle="-.",
    linewidth=1.5,
    color="C1",
    label=r"$\mathbf{s}(\tilde{\mathbf{z}})$",
)
axes[1].text(
    x[60],
    u_before[60] * 0.95,
    "true model state with\n" + r"surrogate solution $\tilde{\mathbf{z}}$",
    color="C1",
    fontsize="small",
    ha="left",
    va="top",
)
axes[1].plot(
    x,
    u_after,
    "C2-",
    linewidth=1.5,
    label=r"$\mathbf{s}(\mathbf{z}')$",
)
axes[1].text(
    x[120],
    u_after.max() + 0.5,
    r"true model state with updated solution $\mathbf{z}'$",
    color="C2",
    fontsize="small",
    ha="center",
    va="bottom",
)
# axes[1].plot(
#     x,
#     u_lofi,
#     "C3:",
#     linewidth=1.5,
#     label=r"$\tilde{\mathbf{s}}(\tilde{\mathbf{z}})$",
# )

axes[1].set_ylim(18, 55)
# axes[1].set_title("States", y=0.9)
axes[1].set_xlabel(r"$x$")
axes[1].set_ylabel(r"State")
axes[1].legend(
    loc="center right",
    bbox_to_anchor=(1, 0.5),
    bbox_transform=fig.transFigure,
)

for ax in axes:
    ax.set_xlim(x[0], x[-1])
plt.subplots_adjust(wspace=0.25, right=0.875)

savefig("simple_hdsa.pdf", fig)

plt.show()

## Error calculations

In [ ]:
# L2 Norm
z_lofi_error = la.norm(z_lofi - z_hifi) / la.norm(z_hifi)
z_hdsa_error = la.norm(z_after - z_hifi) / la.norm(z_hifi)
u_lofi_error = la.norm(u_before - u_hifi) / la.norm(u_hifi)
u_hdsa_error = la.norm(u_after - u_hifi) / la.norm(u_hifi)

print("L2 NORM RELATIVE ERRORS")
print(f"Control error for LoFi optimization: {z_lofi_error:.4%}")
print(f"Control error for HDSA update:       {z_hdsa_error:.4%}")
print(f"State   error for LoFi optimization: {u_lofi_error:.4%}")
print(f"State   error for HDSA update:       {u_hdsa_error:.4%}")

In [ ]:
# Sup Norm
z_lofi_error = np.abs(z_lofi - z_hifi).max() / np.abs(z_hifi).max()
z_hdsa_error = np.abs(z_after - z_hifi).max() / np.abs(z_hifi).max()
u_lofi_error = np.abs(u_before - u_hifi).max() / np.abs(u_hifi).max()
u_hdsa_error = np.abs(u_after - u_hifi).max() / np.abs(u_hifi).max()

print("L-infinity NORM RELATIVE ERRORS")
print(f"Control error for LoFi optimization: {z_lofi_error:.4%}")
print(f"Control error for HDSA update:       {z_hdsa_error:.4%}")
print(f"State   error for LoFi optimization: {u_lofi_error:.4%}")
print(f"State   error for HDSA update:       {u_hdsa_error:.4%}")

In [ ]:
# L1 Norm
z_lofi_error = np.abs(z_lofi - z_hifi).sum() / np.abs(z_hifi).sum()
z_hdsa_error = np.abs(z_after - z_hifi).sum() / np.abs(z_hifi).sum()
u_lofi_error = np.abs(u_before - u_hifi).sum() / np.abs(u_hifi).sum()
u_hdsa_error = np.abs(u_after - u_hifi).sum() / np.abs(u_hifi).sum()

print("L1 NORM RELATIVE ERRORS")
print(f"Control error for LoFi optimization: {z_lofi_error:.4%}")
print(f"Control error for HDSA update:       {z_hdsa_error:.4%}")
print(f"State   error for LoFi optimization: {u_lofi_error:.4%}")
print(f"State   error for HDSA update:       {u_hdsa_error:.4%}")